# 面试问题：怎样用 Conformal Prediction 给模型输出带覆盖率目标的预测区间或集合？

**一句话回答**：先冻结已训练模型，用独立 calibration set 计算 nonconformity score；取有限样本修正分位数 `ceil((n+1)(1-alpha))`，回归输出 `prediction ± q`，分类输出所有 `1-p_class <= q` 的标签。在 calibration/test 可交换假设下得到边际覆盖率，不保证每个群体/每个 x 条件覆盖，漂移会破坏保证。

下面只用 NumPy 实现 split conformal 回归、分类 prediction set、Mondrian 分组校准、漂移反例和版本制品。

In [ ]:
from dataclasses import dataclass  # 导入本单元所需的依赖。
import hashlib, json, math  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。

SEED90=9001; rng90=np.random.default_rng(SEED90)  # 计算并保存当前步骤的中间状态。
assert SEED90==9001  # 用受控断言验证关键不变量。
assert math.ceil(10.1)==11  # 用受控断言验证关键不变量。
assert hashlib.sha256(b"calibration").hexdigest()!=hashlib.sha256(b"test").hexdigest()  # 用受控断言验证关键不变量。

## 1. 数据切分与交换性假设

train 用于拟合模型，calibration 只用于估计 score 分位数，test 只做最终评估。模型或超参数看过 calibration label 后，再用同一集合做 conformal 会破坏简单 split guarantee。时间序列、用户簇和分布漂移也不满足普通 i.i.d. 可交换性。

这里按独立随机样本构造三段，并显式保存 split ID。

In [ ]:
n_train90,n_cal90,n_test90=800,500,2000  # 计算并保存当前步骤的中间状态。
x_all90=rng90.uniform(-3,3,(n_train90+n_cal90+n_test90,2)); noise_scale90=.3+.15*np.abs(x_all90[:,0]); y_all90=2*x_all90[:,0]-1.2*x_all90[:,1]+rng90.normal(0,noise_scale90)  # 计算并保存当前步骤的中间状态。
splits90={"train":np.arange(0,n_train90),"cal":np.arange(n_train90,n_train90+n_cal90),"test":np.arange(n_train90+n_cal90,len(x_all90))}  # 计算并保存当前步骤的中间状态。
assert sum(map(len,splits90.values()))==len(x_all90)  # 用受控断言验证关键不变量。
assert set(splits90["train"]).isdisjoint(splits90["cal"]) and set(splits90["cal"]).isdisjoint(splits90["test"])  # 用受控断言验证关键不变量。
assert max(splits90["train"])<min(splits90["cal"])<min(splits90["test"])  # 用受控断言验证关键不变量。

## 2. 手写线性回归 baseline

给设计矩阵添加常数列，通过 `solve(XᵀX+λI, Xᵀy)` 拟合 ridge；截距不正则。Conformal 不要求模型正确，它包裹任意固定预测器；模型越好，区间通常越窄，但覆盖率来自 calibration rank 而不是线性假设。

模型训练完成后冻结，不能在 calibration cell 更新权重。

In [ ]:
class Ridge90:  # 定义承载本节状态与行为的数据结构。
    def __init__(self,lam=1e-6): self.lam=lam; self.w=None  # 定义本节可复用的核心函数。
    def _design(self,x): x=np.asarray(x,float); return np.c_[np.ones(len(x)),x]  # 定义本节可复用的核心函数。
    def fit(self,x,y):  # 定义本节可复用的核心函数。
        X=self._design(x); reg=np.eye(X.shape[1])*self.lam; reg[0,0]=0; self.w=np.linalg.solve(X.T@X+reg,X.T@np.asarray(y,float)); return self  # 计算并保存当前步骤的中间状态。
    def predict(self,x):  # 定义本节可复用的核心函数。
        if self.w is None: raise RuntimeError("model_not_fitted")  # 按当前条件选择后续控制路径。
        return self._design(x)@self.w  # 返回当前分支计算出的结果。
model90=Ridge90().fit(x_all90[splits90["train"]],y_all90[splits90["train"]]); train_pred90=model90.predict(x_all90[splits90["train"]])  # 计算并保存当前步骤的中间状态。
assert model90.w.shape==(3,) and np.isfinite(model90.w).all()  # 用受控断言验证关键不变量。
assert np.mean((train_pred90-y_all90[splits90["train"]])**2)<.4  # 用受控断言验证关键不变量。
assert np.allclose(model90.w[1:],[2,-1.2],atol=.08)  # 用受控断言验证关键不变量。

## 3. 有限样本修正分位数

calibration score 为绝对残差 `|y-f(x)|`。目标错误率 alpha 时取排序后第 `k=ceil((n+1)(1-alpha))` 个（1-based），并裁到 n。不能直接用默认线性插值 quantile；那通常不是 conformal 的 order statistic。

alpha 太小且 calibration 太少时 k=n，区间由最大残差决定，提示需要更多校准样本。

In [ ]:
def conformal_quantile90(scores,alpha):  # 定义本节可复用的核心函数。
    s=np.sort(np.asarray(scores,float))  # 计算并保存当前步骤的中间状态。
    if s.ndim!=1 or len(s)<2 or not 0<alpha<1 or not np.isfinite(s).all(): raise ValueError("quantile_contract")  # 按当前条件选择后续控制路径。
    k=min(len(s),math.ceil((len(s)+1)*(1-alpha))); return float(s[k-1]),k  # 计算并保存当前步骤的中间状态。
cal_idx90=splits90["cal"]; cal_scores90=np.abs(y_all90[cal_idx90]-model90.predict(x_all90[cal_idx90])); q90,k90=conformal_quantile90(cal_scores90,.1)  # 计算并保存当前步骤的中间状态。
assert 0<q90<2 and k90==math.ceil((n_cal90+1)*.9)  # 用受控断言验证关键不变量。
assert q90 in cal_scores90  # 用受控断言验证关键不变量。
try: conformal_quantile90([],0.1); raise AssertionError("empty scores accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="quantile_contract"  # 捕获预期异常并验证失败分支。

## 4. 回归区间与边际覆盖率

对 test prediction 输出 `[ŷ-q, ŷ+q]`。边际覆盖是随机 test 点落入区间的长期比例，有限样本有波动；不是每个 x 都恰好 90%。本例噪声随 `|x0|` 增大，统一宽度会在中心过宽、边缘欠覆盖，展示条件覆盖缺口。

同时报告平均宽度，覆盖率可通过无限放宽区间轻易“作弊”。

In [ ]:
test_idx90=splits90["test"]; pred90=model90.predict(x_all90[test_idx90]); lower90=pred90-q90; upper90=pred90+q90; covered90=(y_all90[test_idx90]>=lower90)&(y_all90[test_idx90]<=upper90)  # 计算并保存当前步骤的中间状态。
coverage90=float(covered90.mean()); width90=float(np.mean(upper90-lower90)); edge90=np.abs(x_all90[test_idx90,0])>2; center90=np.abs(x_all90[test_idx90,0])<.7  # 计算并保存当前步骤的中间状态。
assert .87<coverage90<.93 and math.isclose(width90,2*q90)  # 用受控断言验证关键不变量。
assert covered90[center90].mean()>covered90[edge90].mean()  # 用受控断言验证关键不变量。
assert np.all(lower90<=upper90) and np.isfinite([coverage90,width90]).all()  # 用受控断言验证关键不变量。

## 5. Mondrian/分组 conformal

若已知噪声群体，可在每组 calibration score 内分别取分位数，再给同组 test 使用；每组样本仍需足够，且分组规则必须在看 test label 前固定。它改善 group-conditional coverage，但不能保证组内每个 x。

这里按 `|x0|>1.5` 分组，边缘组获得更宽区间。

In [ ]:
def group90(x): return (np.abs(np.asarray(x)[:,0])>1.5).astype(int)  # 定义本节可复用的核心函数。
cal_group90=group90(x_all90[cal_idx90]); test_group90=group90(x_all90[test_idx90]); q_group90={g:conformal_quantile90(cal_scores90[cal_group90==g],.1)[0] for g in (0,1)}  # 计算并保存当前步骤的中间状态。
q_each90=np.array([q_group90[g] for g in test_group90]); group_covered90=np.abs(y_all90[test_idx90]-pred90)<=q_each90  # 计算并保存当前步骤的中间状态。
group_cov90={g:float(group_covered90[test_group90==g].mean()) for g in (0,1)}  # 计算并保存当前步骤的中间状态。
assert q_group90[1]>q_group90[0]  # 用受控断言验证关键不变量。
assert all(.86<v<.94 for v in group_cov90.values())  # 用受控断言验证关键不变量。
assert np.mean(2*q_each90)<width90*1.1  # 用受控断言验证关键不变量。

## 6. 分类 prediction set

最简单 score 是 `1-p_true`。校准 q 后，对新样本返回所有满足 `1-p_class <= q` 的类别，即 `p_class >= 1-q`。集合可能为空或包含多个类；不能强行只留 argmax，否则覆盖保证消失。概率模型越有区分度，平均集合越小。

这里构造三分类 softmax 概率并验证经验覆盖。

In [ ]:
def softmax90(z): z=np.asarray(z,float); z=z-z.max(axis=1,keepdims=True); e=np.exp(z); return e/e.sum(axis=1,keepdims=True)  # 定义本节可复用的核心函数。
n_cls90=3500; xc90=rng90.normal(size=(n_cls90,3)); W90=np.array([[1.5,-.4,.2],[-.5,1.4,.1],[.1,-.5,1.3]]); probs90=softmax90(xc90@W90.T); yc90=np.array([rng90.choice(3,p=p) for p in probs90]); calc90=np.arange(0,1000); testc90=np.arange(1000,n_cls90)  # 计算并保存当前步骤的中间状态。
scorec90=1-probs90[calc90,yc90[calc90]]; qc90,_=conformal_quantile90(scorec90,.1); sets90=probs90[testc90]>=1-qc90; class_coverage90=float(sets90[np.arange(len(testc90)),yc90[testc90]].mean()); set_size90=sets90.sum(axis=1)  # 计算并保存当前步骤的中间状态。
assert .87<class_coverage90<.93  # 用受控断言验证关键不变量。
assert np.all((set_size90>=0)&(set_size90<=3)) and set_size90.mean()>1  # 用受控断言验证关键不变量。
assert np.allclose(probs90.sum(axis=1),1)  # 用受控断言验证关键不变量。

## 7. 分布漂移会破坏保证

普通 split conformal 依赖 calibration/test 可交换。若上线噪声突然加倍，旧 q 不再覆盖 90%；这不是实现 bug，而是假设失效。应监控 score/coverage proxy、版本和分布，重新校准或采用适合 covariate/time shift 的方法。

无标签时不能直接知道覆盖率，可监控输入漂移、区间宽度和人工抽样；不要声称仍有保证。

In [ ]:
x_shift90=rng90.uniform(-3,3,(2000,2)); y_shift90=2*x_shift90[:,0]-1.2*x_shift90[:,1]+rng90.normal(0,2*(.3+.15*np.abs(x_shift90[:,0]))); pred_shift90=model90.predict(x_shift90); coverage_shift90=float((np.abs(y_shift90-pred_shift90)<=q90).mean())  # 计算并保存当前步骤的中间状态。
assert coverage_shift90<coverage90-.1  # 用受控断言验证关键不变量。
assert 0<coverage_shift90<1  # 用受控断言验证关键不变量。
assert np.mean(np.abs(y_shift90-pred_shift90))>np.mean(np.abs(y_all90[test_idx90]-pred90))  # 用受控断言验证关键不变量。

## 8. Calibration artifact、监控与面试收束

q 不是裸常数：artifact 绑定 base model hash、score 定义、alpha、calibration snapshot/split、group rule、样本数和代码版本。模型、预处理、标签定义或数据分布改变后必须重新验证。线上记录区间/集合大小、拒识率、分组覆盖（标签回流后）与漂移。

完整回答：独立 calibration → nonconformity → 有限样本 quantile → interval/set → coverage+efficiency → Mondrian → exchangeability/shift → artifact/再校准。

In [ ]:
model_sha90=hashlib.sha256(model90.w.tobytes()).hexdigest(); manifest90={"schema":1,"method":"split_conformal_absolute_residual","model_sha256":model_sha90,"alpha":.1,"q":q90,"calibration_n":n_cal90,"split":"cal-v1","groups":{"rule":"abs_x0_gt_1.5","q":q_group90}}  # 计算并保存当前步骤的中间状态。
raw90=json.dumps(manifest90,sort_keys=True,separators=(",",":")); digest90=hashlib.sha256(raw90.encode()).hexdigest()  # 计算并保存当前步骤的中间状态。
assert len(model_sha90)==len(digest90)==64 and manifest90["calibration_n"]==len(cal_scores90)  # 用受控断言验证关键不变量。
assert math.isclose(manifest90["q"],q90) and set(manifest90["groups"]["q"])=={0,1}  # 用受控断言验证关键不变量。
forged90=dict(manifest90,q=q90/2)  # 计算并保存当前步骤的中间状态。
assert hashlib.sha256(json.dumps(forged90,sort_keys=True,separators=(",",":")).encode()).hexdigest()!=digest90  # 用受控断言验证关键不变量。
print({"coverage":round(coverage90,3),"width":round(width90,3),"group_coverage":group_cov90,"shift_coverage":round(coverage_shift90,3)})  # 执行当前语句以推进本节示例。

## 9. 参考与练习

练习：实现 normalized residual score 让宽度随 x 变化；实现 APS/RAPS 分类集合；按 tenant 做 Mondrian 并处理小样本；模拟时间序列 rolling calibration；比较模型升级前后效率。

参考：[Conformal Prediction 教程](https://arxiv.org/abs/2107.07511)、[Distribution-Free Predictive Inference](https://arxiv.org/abs/1604.04173)、[MAPIE/Conformal 理论参考文档](https://mapie.readthedocs.io/en/stable/theoretical_description_regression.html)。代码从公式实现核心步骤，引用链接只用于继续阅读。